# 02: ML Walk-Forward Training & Model Comparison
Experiments comparing Random Forest vs Sequential LSTM direction classifiers using Walk-Forward temporal validation.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'backend'))

from app.database.db import SessionLocal, init_db
from app.ingestion.price_ingestion import get_price_history_dataframe, fetch_and_store_price_history
from app.ml.models.baseline_rf import RandomForestDirectionClassifier
from app.ml.models.lstm_model import LSTMDirectionClassifier
from app.ml.backtester import run_backtest_simulation

init_db()
db = SessionLocal()

In [ ]:
symbol = 'TCS.NS'
fetch_and_store_price_history(symbol, db, period='2y')
df = get_price_history_dataframe(symbol, db)

rf = RandomForestDirectionClassifier()
rf_metrics = rf.train_walk_forward(df)
rf_pred = rf.predict_latest(df)
print('Random Forest Walk-Forward Results:', rf_metrics)
print('Random Forest Latest Prediction:', rf_pred)

In [ ]:
lstm = LSTMDirectionClassifier(sequence_length=15)
lstm_metrics = lstm.train_walk_forward(df)
lstm_pred = lstm.predict_latest(df)
print('LSTM Walk-Forward Results:', lstm_metrics)
print('LSTM Latest Prediction:', lstm_pred)

In [ ]:
bt = run_backtest_simulation(df, rf)
print(f"Strategy Return: {bt['strategy_return_pct']}% | Benchmark Return: {bt['benchmark_return_pct']}%")
print(f"Strategy Sharpe: {bt['strategy_sharpe']} | Max Drawdown: {bt['max_drawdown_pct']}%")